# Phân Tích Chuyên Sâu COCO vs Pascal VOC 2012
Notebook này tập trung giải quyết các vấn đề:
1. Điều tra tại sao mAP trên VOC lại bằng 0.0.
2. Tìm các lớp dễ bị nhầm lẫn nhất từ Confusion Matrix trên COCO và trực quan hoá các feature của chúng xem có overlap không.
3. Trực quan hoá sự khác biệt phân phối đặc trưng (Domain Shift) giữa tập dữ liệu COCO và Pascal VOC.

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn matplotlib seaborn pyyaml tqdm umap-learn -q
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
REPO_URL = 'https://github.com/Thinh59/ECAAL.git'
REPO_DIR = Path('/kaggle/working/ECAAL')
if REPO_DIR.exists():
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

In [ ]:
import shutil
OUTPUTS_DIR = Path('/kaggle/working/outputs')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = Path('/kaggle/input/datasets/thinhha59/models')
RESULTS_DIR  = DATASET_ROOT / 'results'
SRC_SUBSET   = RESULTS_DIR / 'data' / 'coco_subset'
SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
SUBSET_DIR.mkdir(parents=True, exist_ok=True)
if RESULTS_DIR.exists():
    for fname in ['subset_train_ids.json', 'subset_val_ids.json', 'subset_test_ids.json']:
        if (SRC_SUBSET / fname).exists():
            shutil.copy2(SRC_SUBSET / fname, SUBSET_DIR / fname)
    exp = 'exp_C_efficientnet_cbam_asl'
    src = RESULTS_DIR / exp
    dst = OUTPUTS_DIR / exp
    dst.mkdir(parents=True, exist_ok=True)
    for fname in ['best.pth', 'log.json']:
        if (src / fname).exists(): shutil.copy2(src / fname, dst / fname)

In [ ]:
coco_root = '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017'
voc_root = '/kaggle/input/pascal-voc-2012/VOC2012'
if os.path.exists(coco_root): print('COCO 2017 dataset found')
if os.path.exists(voc_root): print('VOC 2012 dataset found')
else: print('Vui lòng Add Data -> pascal-voc-2012 (kích thước khoảng ~2GB)')

In [ ]:
from dataset import COCOMultiLabelDataset, VOCMultiLabelDataset, get_val_transform
from torch.utils.data import DataLoader
import json
transform = get_val_transform(224)
test_ids = []
test_ids_file = SUBSET_DIR / 'subset_test_ids.json'
if test_ids_file.exists(): test_ids = json.load(open(test_ids_file))
coco_test = COCOMultiLabelDataset(coco_root, split='val', transform=transform, subset_ids=test_ids)
coco_loader = DataLoader(coco_test, batch_size=32, num_workers=2, shuffle=False)
try:
    voc_val = VOCMultiLabelDataset(voc_root, split='val', transform=transform)
    voc_loader = DataLoader(voc_val, batch_size=32, num_workers=2, shuffle=False)
except Exception as e:
    print('Lỗi load VOC:', e)

## 2. Điều tra lỗi mAP VOC = 0.0
Kiểm tra nhãn Ground Truth của VOC xem có bằng 0 hết không? 
Kiểm tra class mapping giữa COCO và VOC xem có đúng không?

In [ ]:
if 'voc_val' in locals():
    print("Tổng số ảnh trong VOC val:", len(voc_val))
    positive_labels = 0
    for img, label in voc_val:
        positive_labels += label.sum().item()
    print("Tổng số nhãn dương trong VOC val:", positive_labels)
    if positive_labels == 0:
        print("!!! CẢNH BÁO: Tập VOC không chứa bất kỳ nhãn dương nào. Đây là nguyên nhân khiến mAP = 0.0!")
        print("Lý do: File annotation của VOC (.txt) có thể chứa khoảng trắng hoặc format khác với code parser.")
    else:
        print("Labels của VOC bình thường. Nguyên nhân mAP = 0.0 có thể do COCO_CLASSES không khớp.")

    # Kiểm tra mapping
    from cross_evaluate import get_coco_to_voc_mapping, VOC_CLASSES
    mapping = get_coco_to_voc_mapping()
    print("\nMapping từ VOC index sang COCO index:")
    for voc_idx, coco_idx in mapping.items():
        print(f"VOC: {VOC_CLASSES[voc_idx]} -> COCO: ID {coco_idx}")
    if len(mapping) < 20:
        print(f"\n!!! CẢNH BÁO: Chỉ ánh xạ được {len(mapping)}/20 lớp. Các lớp bị thiếu sẽ khiến mAP giảm mạnh!")

In [ ]:
from models import build_model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_C = build_model({'backbone': 'efficientnet_b0', 'use_cbam': True, 'num_classes': 80}).to(device)
ckpt_path = OUTPUTS_DIR / 'exp_C_efficientnet_cbam_asl' / 'best.pth'
if ckpt_path.exists():
    model_C.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True)['model_state_dict'])
model_C.eval()
features_dict = {}
def hook_fn(m, i, o):
    features_dict['feat'] = o.detach().cpu().flatten(1)
model_C.gap.register_forward_hook(hook_fn)

In [ ]:
from tqdm.auto import tqdm
def extract_features(loader):
    all_feats, all_targets, all_preds = [], [], []
    with torch.no_grad():
        for imgs, targets in tqdm(loader, leave=False):
            if isinstance(targets, tuple): targets = targets[0]
            out = torch.sigmoid(model_C(imgs.to(device)))
            all_preds.append(out.cpu())
            all_feats.append(features_dict['feat'])
            all_targets.append(targets)
    return {'feats': torch.cat(all_feats).numpy(), 'targets': torch.cat(all_targets).numpy(), 'preds': torch.cat(all_preds).numpy()}
print("Extracting features for COCO Test...")
coco_res = extract_features(coco_loader)
if 'voc_loader' in locals():
    print("Extracting features for VOC Val...")
    voc_res = extract_features(voc_loader)

## 4. Lớp nhập nhằng trên COCO và Visualization

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix
preds_coco_bin = (coco_res['preds'] > 0.5).astype(int)
targets_coco = coco_res['targets'].astype(int)
mcm = multilabel_confusion_matrix(targets_coco, preds_coco_bin)
errors_per_class = mcm[:, 0, 1] + mcm[:, 1, 0]
top_err_idx = np.argsort(errors_per_class)[::-1][:5]
with open(coco_root + '/annotations/instances_val2017.json') as f:
    coco_anns = json.load(f)
cats_sorted = sorted(coco_anns['categories'], key=lambda c: c['id'])
coco_class_names = [c['name'] for c in cats_sorted]
print("Top 5 lớp có số lỗi nhiều nhất trong COCO Test:")
for idx in top_err_idx:
    print(f"- {coco_class_names[idx]} (Lỗi: {errors_per_class[idx]})")

c1, c2 = top_err_idx[0], top_err_idx[1]
print(f"\nTrực quan hoá Feature Space của 2 lớp: {coco_class_names[c1]} và {coco_class_names[c2]}")
idx_c1 = np.where(targets_coco[:, c1] == 1)[0]
idx_c2 = np.where(targets_coco[:, c2] == 1)[0]
idx_both = np.intersect1d(idx_c1, idx_c2)
idx_c1_only = np.setdiff1d(idx_c1, idx_both)
idx_c2_only = np.setdiff1d(idx_c2, idx_both)

feats_c1 = coco_res['feats'][idx_c1_only]
feats_c2 = coco_res['feats'][idx_c2_only]
X_confused = np.concatenate([feats_c1, feats_c2], axis=0)
y_confused = np.array([coco_class_names[c1]] * len(feats_c1) + [coco_class_names[c2]] * len(feats_c2))

from sklearn.decomposition import PCA
pca_confused = PCA(n_components=2)
X_pca_confused = pca_confused.fit_transform(X_confused)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca_confused[:, 0], y=X_pca_confused[:, 1], hue=y_confused, alpha=0.7)
plt.title(f'PCA Feature Overlap: {coco_class_names[c1]} vs {coco_class_names[c2]}')
plt.show()
print("Nhận xét: Nếu overlap lớn, mô hình chưa trích xuất được đặc trưng đủ tốt để phân biệt 2 lớp. Bản chất Classification Head là nhân ma trận trọng số W, nếu overlap nghĩa là không thể dùng 1 siêu phẳng chia tách không gian tuyến tính.")

## 5. Domain Shift Analysis (COCO vs VOC)
Phân tích xem tập VOC có chung phân bố (domain) với COCO hay không.

In [ ]:
if 'voc_res' in locals():
    np.random.seed(42)
    coco_sample_idx = np.random.choice(len(coco_res['feats']), min(2000, len(coco_res['feats'])), replace=False)
    voc_sample_idx = np.random.choice(len(voc_res['feats']), min(2000, len(voc_res['feats'])), replace=False)
    X_domain = np.concatenate([coco_res['feats'][coco_sample_idx], voc_res['feats'][voc_sample_idx]], axis=0)
    y_domain = np.array(['COCO Test'] * len(coco_sample_idx) + ['VOC Val'] * len(voc_sample_idx))
    pca_domain = PCA(n_components=2)
    X_pca_domain = pca_domain.fit_transform(X_domain)
    plt.figure(figsize=(10, 8))
    sns.scatterplot(x=X_pca_domain[:, 0], y=X_pca_domain[:, 1], hue=y_domain, alpha=0.5, s=15, palette=['blue', 'red'])
    plt.title('Domain Shift Analysis: COCO vs Pascal VOC (PCA)')
    plt.show()
    print("Nhận xét:\n- Nếu đám mây VOC tách rời hẳn: Domain Shift lớn.\n- Nếu chúng hoà quyện: Phân phối tương đồng, lỗi mAP = 0.0 khả năng cao do code đọc nhãn sai.")